In [ ]:
import math
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from pathlib import Path
from matplotlib.patches import Polygon

# Fallback for `display()` when this runs outside Jupyter/IPython
try:
    display
except NameError:
    def display(x):
        print(x)

# Input file
gpkg_file = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Output\Masterarbeit\Comparison_NHDA_RA_v2\Comparison_LST_NDVI_DEGURB_CLC_BuildingStructure_Differences_Census.gpkg"

# Load and keep NHDA rows
gdf = gpd.read_file(gpkg_file)
gdf_nhda = gdf[gdf["type"] == "NHDA"].copy()

# -----------------------------------------------------------------------------
# Discover which morphology variables exist, for both the nhda_ and ra_ prefixes
# -----------------------------------------------------------------------------
BASE_MORPHOLOGY_VARS = [
    "built_up_ratio",
    "building_volume_density",
    "building_density",
    "avg_building_height",
]

PRETTY_LABELS = {
    "built_up_ratio": "Built-up Ratio (%)",
    "building_volume_density": "Building Volume Density (m$^3$/m$^2$)",
    "building_density": "Building Density",
    "avg_building_height": "Average Building Height (m)",
    "avg_building_footprint": "Average Building Footprint (m$^2$)",
}


def find_footprint_col(gdf_, prefix):
    """Handles the 'bhilding' typo variant that exists for some prefixes."""
    candidates = [
        f"{prefix}_avg_bhilding_footprint",
        f"{prefix}_avg_building_footprint",
    ]
    for c in candidates:
        if c in gdf_.columns:
            return c
    raise KeyError(
        f"Keine Footprint-Spalte für Prefix '{prefix}' gefunden. Gesucht: {candidates}"
    )


def collect_requested_columns(gdf_, prefix):
    """Returns the list of morphology columns for a given prefix (nhda_ / ra_),
    only including columns that actually exist in the data."""
    cols = [f"{prefix}_{v}" for v in BASE_MORPHOLOGY_VARS if f"{prefix}_{v}" in gdf_.columns]
    try:
        cols.append(find_footprint_col(gdf_, prefix))
    except KeyError as e:
        print(f"Warnung: {e}")
    return cols


PREFIXES = ["nhda", "ra"]
requested_columns_by_prefix = {p: collect_requested_columns(gdf_nhda, p) for p in PREFIXES}

# Flat list of every column we found, across both prefixes
all_requested_columns = [c for cols in requested_columns_by_prefix.values() for c in cols]

if not all_requested_columns:
    raise KeyError("Es wurden weder nhda_* noch ra_* Morphologie-Spalten gefunden.")

# Convert to numeric
for col in all_requested_columns:
    gdf_nhda[col] = pd.to_numeric(gdf_nhda[col], errors="coerce")


def row_label(col, prefix):
    """Pretty row name for a column, e.g. 'nhda_built_up_ratio' -> 'NHDA: Built-up Ratio (%)'."""
    base = col[len(prefix) + 1:]  # strip 'nhda_' / 'ra_'
    if base not in PRETTY_LABELS and base.startswith("avg_b") and base.endswith("footprint"):
        base = "avg_building_footprint"
    pretty = PRETTY_LABELS.get(base, base)
    tag = "NHDA" if prefix == "nhda" else "RA"
    return f"{tag}: {pretty}"


# Build summary table (statistics as columns) for BOTH prefixes
stats_table = pd.DataFrame(index=all_requested_columns)
stats_table["mean"] = gdf_nhda[all_requested_columns].mean()
stats_table["std"] = gdf_nhda[all_requested_columns].std()
stats_table["min"] = gdf_nhda[all_requested_columns].min()
stats_table["p10"] = gdf_nhda[all_requested_columns].quantile(0.10)
stats_table["median"] = gdf_nhda[all_requested_columns].median()
stats_table["p90"] = gdf_nhda[all_requested_columns].quantile(0.90)
stats_table["max"] = gdf_nhda[all_requested_columns].max()
stats_table = stats_table.round(3)

pretty_index = {}
for prefix, cols in requested_columns_by_prefix.items():
    for c in cols:
        pretty_index[c] = row_label(c, prefix)
stats_table = stats_table.rename(index=pretty_index)

stats_table = stats_table.rename(columns={
    "mean": "Mean",
    "std": "Std.",
    "min": "Min",
    "p10": "P10",
    "median": "Median",
    "p90": "P90",
    "max": "Max",
})

# print("Statistik-Tabelle (NHDA & RA):")
# display(stats_table)


# -----------------------------------------------------------------------------
# Shared map styling helpers for the morphology maps
# -----------------------------------------------------------------------------
TARGET_CRS = "EPSG:25832"
GRID_STEP_M = 50000
MAP_PADDING_M = 10000

OUTPUT_DIR_MORPH = Path(r"C:\Users\agz90fk\Documents\EO4CAM\07_Abbildungen\Masterarbeit\Maps_Morphology")
OUTPUT_DIR_MORPH.mkdir(parents=True, exist_ok=True)

COLOR_THEME = ['#fff0f3', '#ffccd5', '#ffb3c1', '#ff8fa3', '#ff758f', '#ff758f', '#ff4d6d', "#c9184a", "#a4133c", "#800f2f", "#590d22"]
CUSTOM_CMAP = LinearSegmentedColormap.from_list('custom_orange_brown', COLOR_THEME)


def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(round(x / 1000))}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{int(round(y / 1000))}"))
    ax.tick_params(axis="both", which="major", labelsize=10, length=0, colors="#9B9999")

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker="+", s=24, linewidths=1.0, color="#8a8a8a", alpha=0.85, zorder=4, clip_on=True)

def add_north_arrow(ax):
    x = 0.94
    y = 0.88
    width = 0.025
    height = 0.09

    # Black left half
    left_triangle = Polygon(
        [
            (x, y + height),
            (x - width, y),
            (x, y + height * 0.30),
        ],
        closed=True,
        transform=ax.transAxes,
        facecolor="#222222",
        edgecolor="#222222",
        linewidth=1.0,
        zorder=10,
    )

    # White right half
    right_triangle = Polygon(
        [
            (x, y + height),
            (x + width, y),
            (x, y + height * 0.30),
        ],
        closed=True,
        transform=ax.transAxes,
        facecolor="white",
        edgecolor="#222222",
        linewidth=1.0,
        zorder=10,
    )

    ax.add_patch(left_triangle)
    ax.add_patch(right_triangle)

    ax.text(
        x,
        y + height + 0.018,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
        color="#222222",
        zorder=10,
    )

def add_scale_bar(ax):
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0
    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000
    x_start = x1 - span_x * 0.35
    y_start = y0 + span_y * 0.060
    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color="#222222", linewidth=1.3, zorder=8)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color="#222222", linewidth=1.0, zorder=8)
    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, "0", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + segment_len, txt_y, "25", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + bar_len, txt_y, "50 km", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)


def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor("#f1f1f1")
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlabel("Easting (km) - UTM 32N", fontsize=11, color="#555555")
    ax.set_ylabel("Northing (km) - UTM 32N", fontsize=11, color="#555555")
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#636262")
        ax.spines[side].set_linewidth(0.8)


# -----------------------------------------------------------------------------
# Load VG250 landkreis data for the map frame
# -----------------------------------------------------------------------------
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\EO4CAM\3_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = "v_vg250_krs"
gdf_lk = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE).to_crs(TARGET_CRS)
gdf_lk = gdf_lk[gdf_lk.geometry.notna()].copy()
# Keep only Bavarian districts (ARS starts with 09)
gdf_lk = gdf_lk[
    gdf_lk["Regionalschlüssel_ARS"].astype(str).str.startswith("09")
].copy()

gdf_lk = gdf_lk[gdf_lk.geometry.notna()].copy()

# Unique Landkreis identifier (VG250 usually has 'ARS'; fall back to other candidates)
LK_KEY_CANDIDATES = ["ARS", "ags", "AGS", "AGS_0", "krs_code", "kreis_code", "SCHLUESSEL", "KRS", "Regionalschlüssel_ARS"]


def _find_key_col(gdf_, candidates):
    cols_lower = {c.lower(): c for c in gdf_.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None


lk_key_col = _find_key_col(gdf_lk, LK_KEY_CANDIDATES)
if lk_key_col is None:
    raise KeyError(
        "Could not find a district-id column in gdf_lk among "
        f"{LK_KEY_CANDIDATES}. Actual columns are:\n{sorted(gdf_lk.columns.tolist())}"
    )
gdf_lk["lk_code"] = gdf_lk[lk_key_col].astype(str).str.strip()




# -----------------------------------------------------------------------------
# Spatial join: assign each NHDA polygon to the Landkreis containing its centroid
# (there is no shared ARS/district-code column in the NHDA data, so this
# replaces the previous attribute-based merge)
# -----------------------------------------------------------------------------
gdf_nhda = gdf_nhda.to_crs(TARGET_CRS)

nhda_centroids = gdf_nhda[["geometry"]].copy()
nhda_centroids["geometry"] = nhda_centroids.geometry.centroid

joined = gpd.sjoin(
    nhda_centroids,
    gdf_lk[["lk_code", "geometry"]],
    how="left",
    predicate="within",
)
# Guard against a centroid sitting exactly on a shared border (rare, but
# would otherwise duplicate the row during the spatial join)
joined = joined[~joined.index.duplicated(keep="first")]

gdf_nhda["lk_code"] = joined["lk_code"]

n_unmatched = gdf_nhda["lk_code"].isna().sum()
if n_unmatched:
    print(f"Warnung: {n_unmatched} NHDA-Polygone konnten keinem Landkreis zugeordnet werden "
          f"(Zentroid liegt evtl. außerhalb aller Landkreis-Polygone).")


def add_labels_top_districts(ax, gdf_plot, values_col="mean_values", std_col=None, top_n=5, offset_m=30000):
    """Label the top N districts with edge placement and duplicate-name disambiguation."""

    required_cols = {values_col, "geometry"}
    if not required_cols.issubset(gdf_plot.columns):
        return

    gdf_labels = gdf_plot[gdf_plot[values_col].notna()].copy()
    if gdf_labels.empty:
        return

    top_districts = gdf_labels.nlargest(top_n, values_col).copy()
    if top_districts.empty:
        return

    bounds = gdf_plot.total_bounds
    minx, miny, maxx, maxy = bounds
    center_x = (minx + maxx) / 2

    frame_x0 = minx - MAP_PADDING_M
    frame_x1 = maxx + MAP_PADDING_M
    frame_y0 = miny - MAP_PADDING_M
    frame_y1 = maxy + MAP_PADDING_M
    frame_height = frame_y1 - frame_y0

    x_left = frame_x0 + 3000
    x_right = frame_x1 - 3000
    y_min = frame_y0 + 6000
    y_max = frame_y1 - 6000
    min_gap = frame_height * 0.09   # minimum spacing between label centres

    name_col = next((c for c in ["GeografischerName_GEN", "GEN", "county", "lk_name", "NAME", "name"] if c in top_districts.columns), None)
    if name_col is None:
        return

    def _norm_name(name):
        n = str(name).strip()
        n = n.replace("Landkreis ", "").replace("Lkr. ", "").replace("Stadtkreis ", "")
        n = n.replace("Kreisfreie Stadt ", "").replace("Landeshauptstadt ", "").replace("Stadt ", "")
        return n.split(",")[0].strip().lower()

    type_by_idx = {}
    grouped = {}
    for idx, row in gdf_labels.iterrows():
        grouped.setdefault(_norm_name(row[name_col]), []).append((idx, row.geometry.area))

    for items in grouped.values():
        if len(items) <= 1:
            continue
        items_sorted = sorted(items, key=lambda x: x[1])
        for i, (idx, _) in enumerate(items_sorted):
            type_by_idx[idx] = "stadt" if i == 0 else "landkreis"

    def _label_text(idx, row):
        raw = str(row[name_col]).strip()
        base = raw.split(",")[0].strip()
        value = row[values_col]

        if std_col and std_col in row.index and not pd.isna(row[std_col]):
            val_str = f"{value:.2f} ± {row[std_col]:.2f}"
        else:
            val_str = f"{value:.2f}"

        if idx in type_by_idx:
            prefix = "Stadt" if type_by_idx[idx] == "stadt" else "Lkr."
            return f"{prefix} {base}\n({val_str})"

        raw_low = raw.lower()
        if "landkreis" in raw_low or raw_low.startswith("lkr."):
            return f"Lkr. {base}\n({val_str})"
        if "stadt" in raw_low or "landeshauptstadt" in raw_low:
            return f"Stadt {base}\n({val_str})"
        return f"{base}\n({val_str})"

    points = list(top_districts.representative_point())
    items = [(idx, row, p, _label_text(idx, row)) for (idx, row), p in zip(top_districts.iterrows(), points)]

    left_items = sorted([item for item in items if item[2].x <= center_x], key=lambda item: item[2].y, reverse=True)
    right_items = sorted([item for item in items if item[2].x > center_x], key=lambda item: item[2].y, reverse=True)

    def _slot_positions(n, ref_ys):
        center_y = sum(ref_ys) / len(ref_ys)
        total_span = min_gap * (n - 1)
        top = min(center_y + total_span / 2, y_max)
        bottom = max(top - total_span, y_min)
        top = min(bottom + total_span, y_max)
        if n == 1:
            return [max(y_min, min(y_max, center_y))]
        step = (top - bottom) / (n - 1)
        return [top - i * step for i in range(n)]

    def _draw_group(group, side):
        if not group:
            return

        x_text = x_left if side == "left" else x_right
        ha = "left" if side == "left" else "right"
        ref_ys = [item[2].y for item in group]
        slots = _slot_positions(len(group), ref_ys)

        for (idx, row, p, label_text), ty in zip(group, slots):
            ax.annotate(
                label_text,
                xy=(p.x, p.y),
                xycoords="data",
                xytext=(x_text, ty),
                textcoords="data",
                ha=ha,
                va="center",
                fontsize=15,
                color="#1f1f1f",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.72, pad=0.6),
                arrowprops=dict(arrowstyle="-", color="#6f6e6e", lw=0.8),
                zorder=6,
                clip_on=False,
            )

    _draw_group(left_items, "left")
    _draw_group(right_items, "right")


def plot_morphology_map(var_name, meta):
    agg = gdf_nhda.groupby("lk_code")[var_name].agg(mean_values="mean", std_values="std").reset_index()
    map_df = gdf_lk.merge(agg, on="lk_code", how="left")

    values = map_df["mean_values"].dropna()
    if values.empty:
        print(f"Warnung: Keine Werte für {var_name} gefunden - Karte wird übersprungen.")
        return

    vmin = values.min()
    vmax = values.max()
    if vmin == vmax:
        vmax = vmin + 1e-9

    fig, ax = plt.subplots(figsize=(8.8, 9.2))
    add_scientific_frame(ax, gdf_lk)
    map_df.plot(
        ax=ax,
        column="mean_values",
        cmap=CUSTOM_CMAP,
        linewidth=0.35,
        edgecolor="#5a5a5a",
        legend=False,
        vmin=vmin,
        vmax=vmax,
        missing_kwds={"color": "#d9d9d9", "edgecolor": "#b7b7b7", "label": "No data"},
    )

    ax.set_title(f"Mean {meta['label']}", fontsize=18, pad=14)
    add_north_arrow(ax)
    add_scale_bar(ax)

    # add_labels_top_districts(ax, map_df, values_col="mean_values", std_col="std_values", top_n=5, offset_m=30000)

    sm = ScalarMappable(norm=Normalize(vmin=vmin, vmax=vmax), cmap=CUSTOM_CMAP)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.036, pad=0.02)
    cbar.set_label(meta["cbar"], fontsize=12)
    cbar.ax.tick_params(labelsize=11)

    missing_handle = Patch(facecolor="#d9d9d9", edgecolor="#b7b7b7", label="No data")
    ax.legend(handles=[missing_handle], loc="upper left", frameon=True, framealpha=0.95, facecolor="white", edgecolor="#cccccc", fontsize=10)

    out_file = OUTPUT_DIR_MORPH / f"{meta['prefix']}_morphology_{meta['suffix']}_landkreis_mean.jpg"
    plt.savefig(out_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {out_file}")


# -----------------------------------------------------------------------------
# Build the metadata dict for ALL variables (both nhda_ and ra_) and plot them
# -----------------------------------------------------------------------------
morphology_vars = {}
for prefix in PREFIXES:
    tag = "NHDA" if prefix == "nhda" else "RA"
    for col in requested_columns_by_prefix[prefix]:
        base = col[len(prefix) + 1:]
        if base.startswith("avg_b") and base.endswith("footprint"):
            base = "avg_building_footprint"
        pretty = PRETTY_LABELS.get(base, base)
        morphology_vars[col] = {
            "label": f"{tag} {pretty}",
            "cbar": pretty,
            "suffix": base.replace("avg_building_", "avg_"),
            "prefix": prefix,
        }

for var_name, meta in morphology_vars.items():
    if var_name in gdf_nhda.columns:
        plot_morphology_map(var_name, meta)